# AI Agents Workshop — Day 1 Labs (Colab)S4DS KJSIT. Run the setup cell first, then work down.**Before anything:** click the 🔑 key icon in the left sidebar → *Add new secret* →name it exactly `HF_TOKEN` → paste your token from[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) →toggle **Notebook access** on.

## Setup — run this once

In [ ]:
!pip install -q "huggingface_hub>=0.30.0" "smolagents>=1.14.0" "transformers>=4.45.0" duckduckgo-searchimport osfrom google.colab import userdataos.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")MODEL_ID = "Qwen/Qwen2.5-72B-Instruct"from huggingface_hub import InferenceClientclient = InferenceClient(model=MODEL_ID, token=os.environ["HF_TOKEN"])r = client.chat.completions.create(    messages=[{"role": "user", "content": "Reply with exactly: pong"}],    max_tokens=10,)print("model said:", r.choices[0].message.content)print("SETUP OK" if "pong" in r.choices[0].message.content.lower() else "SETUP FAILED")

---## Lab 1 — What the model actually seesAn LLM does not see a list of messages. It sees **one string**.

In [ ]:
from transformers import AutoTokenizertokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")messages = [    {"role": "system", "content": "You are a terse assistant."},    {"role": "user", "content": "What is the capital of Maharashtra?"},    {"role": "assistant", "content": "Mumbai."},    {"role": "user", "content": "And its population?"},]prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)print(repr(prompt))print()print(prompt)

**Try it:** swap in `meta-llama/Llama-3.2-1B-Instruct`. Completely different template.This is why prompts don't transfer between models.

---## Lab 2 — A tool is a function + a description

In [ ]:
import inspectdef get_weather(city: str) -> str:    """Get the current weather for an Indian city.    Args:        city: Name of the city, e.g. "Pune".    """    fake = {"mumbai": "32C, humid", "pune": "27C, clear", "delhi": "38C, hazy"}    return fake.get(city.lower().strip(), f"No weather data for {city}")def describe(func):    sig = inspect.signature(func)    doc = (func.__doc__ or "").strip().split("\n")[0]    return f"- {func.__name__}{sig}: {doc}"print(describe(get_weather))print()print("^ THIS is the only thing the model ever sees about your function.")

**Try it:** delete the docstring and re-run. Your docstring *is* your prompt.

---## Lab 3 — Write the agent loop yourselfThe most important cell in this notebook. Read it line by line before running.

In [ ]:
import reMAX_STEPS = 6def get_weather(city: str) -> str:    fake = {"mumbai": "32C, humid", "pune": "27C, clear", "delhi": "38C, hazy"}    return fake.get(city.lower().strip(), f"No weather data for {city}")def calculate(expression: str) -> str:    allowed = set("0123456789+-*/(). ")    if not set(expression) <= allowed:        return "Error: only numbers and + - * / ( ) allowed."    try:        return str(eval(expression))    except Exception as exc:        return f"Error: {exc}"TOOLS = {"get_weather": get_weather, "calculate": calculate}SYSTEM_PROMPT = """You solve tasks by reasoning step by step and using tools.Available tools:- get_weather(city): current weather for an Indian city.- calculate(expression): evaluate arithmetic.Reply in exactly this format, one step at a time:Thought: <your reasoning>Action: <tool_name>(<single argument>)After each Action you will be shown an Observation.When done, reply with:Thought: <why you can answer now>Final Answer: <your answer>Never write an Observation yourself."""ACTION_RE = re.compile(r"Action:\s*(\w+)\((.*?)\)", re.DOTALL)def run(task):    messages = [        {"role": "system", "content": SYSTEM_PROMPT},        {"role": "user", "content": task},    ]    for step in range(1, MAX_STEPS + 1):        print(f"\n{'-'*50}\nSTEP {step}\n{'-'*50}")        resp = client.chat.completions.create(            messages=messages, max_tokens=400, stop=["Observation:"]        )        out = resp.choices[0].message.content.strip()        print(out)        messages.append({"role": "assistant", "content": out})        if "Final Answer:" in out:            return out.split("Final Answer:", 1)[1].strip()        m = ACTION_RE.search(out)        if not m:            messages.append({"role": "user", "content":                "Invalid format. Use 'Action: tool(arg)' or 'Final Answer: ...'."})            continue        name, arg = m.group(1), m.group(2).strip().strip("\"'")        obs = TOOLS[name](arg) if name in TOOLS else f"Error: no tool '{name}'"        print(f"\nObservation: {obs}")        messages.append({"role": "user", "content": f"Observation: {obs}"})    return "Gave up - hit MAX_STEPS."print(run("What's the weather in Pune, and what is that temperature plus 5?"))

**Try it:** remove `stop=["Observation:"]` and re-run. The model invents its ownobservations and confidently hallucinates. That one argument is the differencebetween an agent and a liar.

---## Lab 4 — The same thing with smolagents

In [ ]:
from smolagents import CodeAgent, ToolCallingAgent, InferenceClientModel, tool@tooldef get_weather(city: str) -> str:    """Get the current weather for an Indian city.    Use this whenever the user asks about temperature, rain, or humidity.    Args:        city: Name of the city, e.g. "Pune".    """    fake = {"mumbai": "32C, humid", "pune": "27C, clear", "delhi": "38C, hazy"}    return fake.get(city.lower().strip(), f"No weather data for {city}")@tooldef get_mess_menu(day: str) -> str:    """Get the hostel mess menu for a day of the week.    Args:        day: Day name, e.g. "Tuesday".    """    menu = {"monday": "Rajma chawal", "tuesday": "Pav bhaji", "wednesday": "Veg biryani"}    return menu.get(day.lower().strip(), f"No menu for {day}")model = InferenceClientModel(model_id=MODEL_ID, token=os.environ["HF_TOKEN"])agent = CodeAgent(tools=[get_weather, get_mess_menu], model=model, max_steps=5)print(agent.run("Weather in Pune and Tuesday's mess menu - good day to eat outside?"))

**Try it:** add `verbosity_level=2` and compare the system prompt smolagentsgenerated against the one you wrote by hand in Lab 3.

---## Lab 5 — Deploy to a SpaceNot a notebook lab. See `day1/05_space/` in the repo:1. New Space → SDK **Gradio** → CPU basic → Public2. Settings → Variables and secrets → add `HF_TOKEN`3. Upload `app.py` and `requirements.txt`4. Share the link---## Project 1See `projects/project-1.md`. Due before Day 2.